In [1]:
import gzip
import json
import os
import sys
from pathlib import Path

from IPython.display import HTML, display

### Build MAOMAO sequence cards

This notebook creates one compact cross-layer card per normalized MAOMAO sequence. The card summarizes final activities, direct-evidence counts, ontology effects, source-specific negative-evidence categories, affected organism or toxicity-target categories supported by direct positive evidence, numerical toxicity properties, and four selected physicochemical descriptors.

Detailed source records are written to companion audit tables. Descriptor values, one-hot encodings, and embedding vectors are not repeated in every card; they remain in their canonical layers and are joined by the stable SHA-256 sequence identifier.

Primary output: `sequence_profiles/sequence_cards.jsonl.gz` in the repository root. The same directory also receives detailed evidence tables, a JSON Schema, metadata, checksums, and JSON/HTML examples.

- Locate the repository

In [2]:
def find_repo_root(start=None):
    configured = start or os.environ.get("MAOMAO_ROOT") or Path.cwd()
    start_path = Path(configured).expanduser().resolve()

    for candidate in (start_path, *start_path.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "maomao").is_dir():
            return candidate

    raise FileNotFoundError(
        f"Could not locate the MAOMAO repository root from {start_path}. "
        "Run this notebook inside the repository or set MAOMAO_ROOT."
    )


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from maomao.sequence_cards import CardConfig, build_sequence_cards

print("Repository root:", REPO_ROOT)

Repository root: /home/nicole/Descargas/maomao


- Configuration

By default the resource version is read from `pyproject.toml`. Optional paths can be supplied to `CardConfig` if a layer is stored elsewhere.

In [3]:
config = CardConfig(
    repo_root=REPO_ROOT,
    # resource_version="1.1.0",
    # descriptors_path=REPO_ROOT / "dataset_characterization" / "sequence_descriptors.csv",
    # numerical_representation_root=REPO_ROOT / "numerical_representation_data" / "maomao",
)

display(config.resolved())

CardConfig(repo_root=PosixPath('/home/nicole/Descargas/maomao'), output_dir=PosixPath('/home/nicole/Descargas/maomao/sequence_profiles'), resource_version='1.1.0', master_path=PosixPath('/home/nicole/Descargas/maomao/processed_data/processed_data/maomao_sequence_pivot.csv'), measurements_path=PosixPath('/home/nicole/Descargas/maomao/processed_data/processed_data/maomao_toxicity_measurements.csv'), negative_evidence_metadata_path=PosixPath('/home/nicole/Descargas/maomao/raw_data/evidence_negative_dataset.xlsx'), toxicity_target_metadata_path=PosixPath('/home/nicole/Descargas/maomao/raw_data/tasks_by_source.xlsx'), descriptors_path=PosixPath('/home/nicole/Descargas/maomao/dataset_characterization/sequence_descriptors.csv'), numerical_representation_root=PosixPath('/home/nicole/Descargas/maomao/numerical_representation_data/maomao'))

- Generate the card layer

In [4]:
result = build_sequence_cards(config)
metadata = result["metadata"]

print("Output directory:", result["output_dir"])
print(f"Cards: {metadata['card_count']:,}")
print("Evidence matches master pivot:", metadata["validation"]["evidence_matches_master_pivot"])
print("Generated files:")
for path in result["files"]:
    print(" -", path.relative_to(result["output_dir"]))

Output directory: /home/nicole/Descargas/maomao/sequence_profiles
Cards: 71,857
Evidence matches master pivot: True
Generated files:
 - sequence_cards.jsonl.gz
 - sequence_activity_evidence.csv.gz
 - sequence_source_evidence.csv.gz
 - sequence_card_schema.json
 - metadata.json
 - README.md
 - examples/sequence_card_example.json
 - examples/sequence_card_example.html
 - CHECKSUMS.sha256
